# DuckLake Streaming Demo (Jupyter)

Mirrors `streaming_demo.py` (marimo). Generate synthetic data with a parametrized row
count, write it to DuckLake via streaming, and query it interactively. Setup and write
are Python; queries use the engine's DuckDB connection directly via `con.execute(...).pl()`.


In [1]:
import sys
from pathlib import Path

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

_REPO = Path.cwd()
if _REPO.name == "notebooks":
    _REPO = _REPO.parent
if str(_REPO / "src") not in sys.path:
    sys.path.insert(0, str(_REPO / "src"))

from ducklake_playground import (
    DuckLakeEngine,
    GeneratorSpec,
    StreamingGenerator,
    load_config,
    measure_time_and_memory,
)

config = load_config(_REPO / "config.yaml")
config.name

'DuckLake Playground'

## Parameters


In [3]:
ROW_COUNT = 1_000_000
TABLE_NAME = "demo_table"
STORAGE_MODE = "s3"
CHUNK_SIZE = config.batch_size
SEED = config.schema.seed

assert STORAGE_MODE in config.storage_modes
f"Will create {TABLE_NAME} with {ROW_COUNT:,} rows in {STORAGE_MODE} storage"

'Will create demo_table with 1,000,000 rows in s3 storage'

In [4]:
engine = DuckLakeEngine()
engine.setup(config, STORAGE_MODE)
con = engine.connection
catalog = engine.catalog_name
fq = f"{catalog}.main.{TABLE_NAME}"
fq

2026-05-09 18:42:11.151 | INFO     | ducklake_playground.engine:setup:142 - DuckLake engine setup complete (storage=s3, data_path=s3://warehouse/ducklake/, pg_baseline=8957.7 KB)


'playground_ducklake_s3.main.demo_table'

## Write-time options

Persisted in the Postgres catalog. Set BEFORE writing.


In [5]:
con.execute(f"CALL {catalog}.set_option('parquet_version', 2)")
con.execute(f"CALL {catalog}.set_option('parquet_compression', 'zstd')")
con.execute(f"CALL {catalog}.set_option('parquet_row_group_size_bytes', '16MB')")
"options set"

'options set'

## Stream-generate + write


In [6]:
gen = StreamingGenerator(
    GeneratorSpec(
        schema_config=config.schema,
        total_rows=ROW_COUNT,
        chunk_size=CHUNK_SIZE,
        seed=SEED,
    )
)
with measure_time_and_memory() as t:
    engine.write_overwrite(TABLE_NAME, gen.arrow_reader(), gen.schema)
timing = t[0]
print(
    f"Wrote {ROW_COUNT:,} rows in {timing.wall_time_seconds:.2f}s | "
    f"peak RSS {timing.peak_rss_mb:.0f} MB (delta {timing.delta_rss_mb:.0f} MB)"
)

Wrote 1,000,000 rows in 9.11s | peak RSS 1187 MB (delta 1069 MB)


## Schema + sanity counts


In [7]:
con.execute(f"DESCRIBE {fq}").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""id""","""BIGINT""","""YES""",null,"""NULL""",null
"""event_date""","""DATE""","""YES""",null,"""NULL""",null
"""int8_col""","""TINYINT""","""YES""",null,"""NULL""",null
"""int16_col""","""SMALLINT""","""YES""",null,"""NULL""",null
"""int32_col""","""INTEGER""","""YES""",null,"""NULL""",null
…,…,…,…,…,…
"""text_col""","""VARCHAR""","""YES""",null,"""NULL""",null
"""bool_col""","""BOOLEAN""","""YES""",null,"""NULL""",null
"""list_col""","""INTEGER[]""","""YES""",null,null,null


In [8]:
con.execute(
    f"""
    SELECT COUNT(*) AS n,
           MIN(event_date) AS min_d,
           MAX(event_date) AS max_d,
           COUNT(DISTINCT event_date) AS distinct_partitions
    FROM {fq}
    """
).pl()

n,min_d,max_d,distinct_partitions
i64,date,date,i64
1000000,2024-01-01,2024-01-30,30


## Query


In [9]:
query = f"""
SELECT varchar_col,
       COUNT(*) AS cnt,
       SUM(int64_col) AS sum_val,
       AVG(float64_col) AS avg_val,
       MIN(event_date) AS min_date,
       MAX(event_date) AS max_date
FROM {fq}
WHERE event_date BETWEEN DATE '2024-01-10' AND DATE '2024-01-20'
GROUP BY varchar_col
ORDER BY cnt DESC
LIMIT 20
"""
with measure_time_and_memory() as t:
    result = con.execute(query).pl()
timing = t[0]
print(
    f"{result.height:,} rows in {timing.wall_time_seconds:.3f}s | "
    f"peak RSS {timing.peak_rss_mb:.0f} MB (delta {timing.delta_rss_mb:.0f} MB)"
)
result

20 rows in 0.165s | peak RSS 154 MB (delta 17 MB)


varchar_col,cnt,sum_val,avg_val,min_date,max_date
str,i64,"decimal[38,0]",f64,date,date
"""value_419""",433,-5929037243322729371,4.6756e13,2024-01-10,2024-01-20
"""value_676""",424,-8371860006768744814,-3.0823e12,2024-01-10,2024-01-20
"""value_077""",420,-96088639395365615433,4.4505e13,2024-01-10,2024-01-20
"""value_246""",417,-166182687143014097103,6.7794e12,2024-01-10,2024-01-20
"""value_484""",416,-127368783807082808305,3.4877e13,2024-01-10,2024-01-20
…,…,…,…,…,…
"""value_313""",408,-77241483110514276968,-1.5637e13,2024-01-10,2024-01-20
"""value_381""",408,89552020945952692861,1.9696e13,2024-01-10,2024-01-20
"""value_174""",407,-87787349091713488981,-1.0718e13,2024-01-10,2024-01-20


## Snapshots + maintenance


In [10]:
con.execute(f"SELECT * FROM {catalog}.snapshots() ORDER BY snapshot_id DESC LIMIT 10").pl()

snapshot_id,snapshot_time,schema_version,changes,author,commit_message,commit_extra_info
i64,"datetime[μs, Europe/Rome]",i64,list[struct[2]],str,str,str
35,2026-05-09 18:42:24.284584 CEST,23,"[{""tables_inserted_into"",[""5""]}]",null,null,null
34,2026-05-09 18:42:24.217542 CEST,23,"[{""tables_altered"",[""5""]}]",null,null,null
33,2026-05-09 18:42:24.188056 CEST,22,"[{""tables_created"",[""main.demo_table""]}]",null,null,null
32,2026-05-09 18:42:23.730337 CEST,21,"[{""tables_dropped"",[""3""]}]",null,null,null
31,2026-05-09 18:40:40.124998 CEST,20,"[{""tables_inserted_into"",[""3""]}, {""inlined_insert"",[""3""]}, {""inlined_delete"",[""3""]}]",null,null,null
30,2026-05-09 18:40:38.944936 CEST,20,"[{""tables_altered"",[""3""]}]",null,null,null
29,2026-05-09 18:40:38.555364 CEST,19,"[{""tables_altered"",[""3""]}]",null,null,null
28,2026-05-09 18:40:35.800480 CEST,18,"[{""tables_altered"",[""3""]}]",null,null,null
27,2026-05-09 18:40:30.168753 CEST,17,"[{""inlined_insert"",[""3""]}]",null,null,null


In [12]:
# Run after several writes to compact files. Order matters.
con.execute(f"CALL ducklake_merge_adjacent_files('{catalog}')")
con.execute(f"CALL ducklake_expire_snapshots('{catalog}', older_than => now() - INTERVAL '7 days')")
con.execute(f"CALL ducklake_cleanup_old_files('{catalog}', cleanup_all => true)")

In [ ]:
# Drop the demo table when finished. Comment out to keep iterating.
# engine.teardown(TABLE_NAME)
# engine.close()